# Tema 3 - Analisis Dinamico de Mecanismos Planos

**Asignatura:** Teoria de Maquinas y Mecanismos

---

### Objetivos

1. Clasificar las acciones (fuerzas y pares) sobre un mecanismo plano
2. Plantear el Diagrama de Cuerpo Libre (DCL) de cada eslabon
3. Aplicar el **Principio de D'Alembert** (metodo vectorial) para resolver el equilibrio dinamico
4. Aplicar el **Principio de Potencias Virtuales** (metodo analitico) como alternativa mas eficiente
5. Distinguir entre **analisis inverso** (movimiento dado -> par motor) y **analisis directo** (fuerzas dadas -> movimiento)
6. Utilizar el **Teorema de las Fuerzas Vivas** para balances energeticos
7. Resolver problemas de **equilibrado de rotores**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch, Arc, Circle
from matplotlib.lines import Line2D
from scipy.optimize import fsolve
from scipy.integrate import solve_ivp

# ── Paleta de colores unificada ──
COLOR_PRINCIPAL = '#2171b5'   # Azul - barras, curvas principales
COLOR_FIJO      = '#636363'   # Gris oscuro - barra fija / suelo
COLOR_PUNTO     = '#238b45'   # Verde - puntos de operacion, resultados
COLOR_ROJO      = '#cb181d'   # Rojo - fuerzas, errores, alertas
COLOR_NARANJA   = '#ff7f00'   # Naranja - secundario
COLOR_MORADO    = '#6a3d9a'   # Morado - terciario
COLOR_CLARO     = '#a6cee3'   # Azul claro - rellenos

# ── Helpers de dibujo ──
def draw_link(ax, p1, p2, color=COLOR_PRINCIPAL, lw=5, zorder=2):
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color=color,
            lw=lw, solid_capstyle='round', zorder=zorder)

def draw_joint(ax, p, fixed=False, color='black', ms=10, zorder=5):
    if fixed:
        ax.plot(*p, 's', color=color, ms=ms, zorder=zorder)
    else:
        ax.plot(*p, 'o', color=color, ms=ms, zorder=zorder,
                markerfacecolor='white', markeredgewidth=2)

def draw_ground(ax, center, width, y_offset=-0.15):
    x0 = center[0] - width / 2
    y0 = center[1] + y_offset
    for xi in np.linspace(x0, x0 + width, 8):
        ax.plot([xi, xi - 0.12], [y0, y0 - 0.18], '-', color=COLOR_FIJO, lw=1)
    ax.plot([x0, x0 + width], [y0, y0], '-', color=COLOR_FIJO, lw=2)

def draw_force_arrow(ax, start, end, color=COLOR_ROJO, lw=2, hw=0.12, hl=0.15):
    dx, dy = end[0] - start[0], end[1] - start[1]
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))

def clean_ax(ax, title=''):
    ax.set_aspect('equal')
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, fontweight='bold')

## 1. Formulario del Tema

### Fuerzas de inercia

$$\boxed{\vec{F}_i = -m\,\vec{a}_G}$$

$$\boxed{M_i = -I_G\,\alpha}$$

$$\boxed{I_O = I_G + m\,d^2 \quad \text{(Steiner)}}$$

### Rozamiento seco (Amontons-Coulomb)

$$\boxed{F_{\text{max}} = \mu_s\,N \quad (\text{estatico})}$$

$$\boxed{T = \mu_d\,N \quad (\text{dinamico}), \quad \mu_d < \mu_s}$$

### Principio de D'Alembert (equilibrio de cada eslabon)

$$\boxed{\sum F_x = 0, \quad \sum F_y = 0, \quad \sum M_O = 0 \quad \text{(incluyendo } \vec{F}_i, M_i\text{)}}$$

### Principio de Potencias Virtuales

$$\boxed{P_{\text{motriz}} + P_{\text{grav}} + P_{\text{inercia}} + P_{\text{pasivas}} = 0}$$

$$P = \vec{F} \cdot \vec{v}, \quad P = M \cdot \omega$$

### Teorema de las Fuerzas Vivas

$$\boxed{W_{\text{total}} = \Delta T = T_f - T_i}$$

$$T = \frac{1}{2}m v_G^2 + \frac{1}{2}I_G\,\omega^2$$

### Trabajo de fuerzas tipicas

| Fuerza | Trabajo |
|:-------|:--------|
| Motriz / Resistente | $W = \int \vec{F} \cdot d\vec{r} = \int M\,d\theta$ |
| Gravedad | $\Delta W_g = -mg\,\Delta z = -\Delta E_{pg}$ |
| Inercia | $\Delta W_{in} = -\tfrac{1}{2}m\,\Delta(v^2) = -\Delta T$ |
| Elastica (interna) | $W = 0$ |

### Equilibrado de rotores

$$\boxed{\sum m_i\,r_i\,e^{j\theta_i} = 0 \quad \text{(estatico)}}$$

$$\boxed{\sum m_i\,r_i\,z_i\,e^{j\theta_i} = 0 \quad \text{(dinamico: momentos)}}$$

## 2. Tipos de acciones sobre un mecanismo

Las fuerzas y pares que actuan sobre un mecanismo se clasifican en **externas** e **internas**.

**Fuerzas externas:**
- **Motrices:** aportan energia al sistema ($W_{\text{motriz}} > 0$)
- **Resistentes utiles:** realizan trabajo util ($W_{\text{util}} < 0$ desde el punto de vista del mecanismo)
- **Resistentes pasivas:** disipan energia (rozamiento, etc.)
- **Gravedad:** conservativa, $\Delta W_g = -mg\,\Delta z$
- **Inercia:** $\Delta W_{in} = -\tfrac{1}{2}m\,\Delta(v^2)$

**Fuerzas internas:**
- **Esfuerzos internos:** en barras rigidas el trabajo es nulo (elasticas $W=0$, inelasticas $\rightarrow$ disipacion)
- **Reacciones en pares cinematicos:** $\vec{R}_{ij}^A = -\vec{R}_{ji}^A$ (accion-reaccion)

In [ ]:
# ── Diagrama de clasificacion de fuerzas ──
fig, ax = plt.subplots(figsize=(12, 7))
clean_ax(ax)
ax.set_xlim(-0.5, 12)
ax.set_ylim(-0.5, 8.5)

# Titulo
ax.text(6, 8, 'Clasificacion de Acciones', fontsize=16, ha='center',
        fontweight='bold', color='black')

# Caja principal
boxes = [
    # Externas
    (0.3, 6.0, 5.0, 1.2, 'FUERZAS EXTERNAS', COLOR_ROJO, 0.15),
    (0.3, 4.2, 2.2, 1.2, 'Motrices\n$W_{motriz}>0$', COLOR_ROJO, 0.10),
    (2.8, 4.2, 2.5, 1.2, 'Resistentes\n(utiles + pasivas)', COLOR_NARANJA, 0.10),
    (0.3, 2.4, 2.2, 1.2, 'Gravedad\n$\\Delta W_g=-mg\\Delta z$', COLOR_MORADO, 0.10),
    (2.8, 2.4, 2.5, 1.2, 'Inercia\n$F_i=-ma_G$', COLOR_PRINCIPAL, 0.10),
    # Internas
    (6.3, 6.0, 5.2, 1.2, 'FUERZAS INTERNAS', COLOR_PUNTO, 0.15),
    (6.3, 4.2, 2.3, 1.2, 'Esfuerzos\ninternos\n(W=0 rigido)', COLOR_PUNTO, 0.10),
    (9.0, 4.2, 2.5, 1.2, 'Reacciones\nen pares\n$R_{ij}=-R_{ji}$', COLOR_FIJO, 0.10),
]

for (x, y, w, h, txt, col, alpha) in boxes:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                          facecolor=col, alpha=alpha, edgecolor=col, lw=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, txt, ha='center', va='center',
            fontsize=10, fontweight='bold', color=col)

# Flechas de conexion (padre->hijos)
for (x_start, y_start, x_end, y_end) in [
    (2.8, 6.0, 1.4, 5.4), (2.8, 6.0, 4.05, 5.4),   # ext -> motrices, resistentes
    (2.8, 4.2, 1.4, 3.6), (2.8, 4.2, 4.05, 3.6),   # ext -> gravedad, inercia
    (8.9, 6.0, 7.45, 5.4), (8.9, 6.0, 10.25, 5.4),  # int -> esfuerzos, reacciones
]:
    ax.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#333'))

# Balance energetico
ax.text(6, 1.0, r'$\sum W = W_{motriz} + W_{util} + W_{grav} + W_{inercia} + W_{pasivas} = 0$',
        fontsize=13, ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=COLOR_CLARO, alpha=0.4, edgecolor=COLOR_PRINCIPAL, lw=2))

plt.tight_layout()
plt.show()

## 3. Fuerzas de inercia

Segun el **Principio de D'Alembert**, el movimiento de un solido rigido se puede tratar como un problema
 de equilibrio estatico si se anaden unas fuerzas ficticias llamadas **fuerzas de inercia**:

- **Fuerza de inercia** (aplicada en el centro de masas $G$):
$$\vec{F}_i = -m\,\vec{a}_G$$

- **Momento de inercia** (respecto a $G$):
$$M_i = -I_G\,\alpha$$

Se pueden trasladar a un punto $O$ cualquiera usando el **Teorema de Steiner**:

$$M_O = -(I_G + m\,|OG|^2)\,\alpha = -I_O\,\alpha$$

donde $I_O = I_G + m\,d^2$ es el momento de inercia respecto al punto $O$.

In [ ]:
# ── DCL de un eslabon con fuerzas de inercia ──
fig, ax = plt.subplots(figsize=(9, 6))
clean_ax(ax, 'Diagrama de Cuerpo Libre - Eslabon con fuerzas de inercia')
ax.set_xlim(-1, 9)
ax.set_ylim(-2, 5)

# Eslabon (barra)
A = np.array([1.0, 1.0])
B = np.array([7.0, 3.0])
G = (A + B) / 2  # centro de masas
draw_link(ax, A, B, color=COLOR_PRINCIPAL, lw=8)
draw_joint(ax, A)
draw_joint(ax, B)

# Centro de masas
ax.plot(*G, 'D', color=COLOR_NARANJA, ms=10, zorder=6)
ax.text(G[0]+0.3, G[1]+0.3, '$G$', fontsize=14, fontweight='bold', color=COLOR_NARANJA)

# Etiquetas A, B
ax.text(A[0]-0.5, A[1]-0.2, '$A$', fontsize=14, fontweight='bold')
ax.text(B[0]+0.3, B[1]-0.2, '$B$', fontsize=14, fontweight='bold')

# Reacciones en A: R_Ax, R_Ay
draw_force_arrow(ax, A, (A[0]+1.2, A[1]), color=COLOR_PUNTO)
ax.text(A[0]+1.3, A[1]+0.1, '$R_{Ax}$', fontsize=12, color=COLOR_PUNTO)
draw_force_arrow(ax, A, (A[0], A[1]+1.2), color=COLOR_PUNTO)
ax.text(A[0]+0.15, A[1]+1.3, '$R_{Ay}$', fontsize=12, color=COLOR_PUNTO)

# Reacciones en B: R_Bx, R_By
draw_force_arrow(ax, B, (B[0]+1.2, B[1]), color=COLOR_PUNTO)
ax.text(B[0]+1.3, B[1]+0.1, '$R_{Bx}$', fontsize=12, color=COLOR_PUNTO)
draw_force_arrow(ax, B, (B[0], B[1]+1.2), color=COLOR_PUNTO)
ax.text(B[0]+0.15, B[1]+1.3, '$R_{By}$', fontsize=12, color=COLOR_PUNTO)

# Fuerza de inercia F_i = -m*a_G (en G)
draw_force_arrow(ax, G, (G[0]-1.5, G[1]-0.5), color=COLOR_ROJO)
ax.text(G[0]-2.2, G[1]-0.7, '$\\vec{F}_i = -m\\vec{a}_G$', fontsize=12,
        color=COLOR_ROJO, fontweight='bold')

# Momento de inercia M_i (arco)
arc = Arc(G, 1.5, 1.5, angle=0, theta1=30, theta2=300, color=COLOR_ROJO, lw=2)
ax.add_patch(arc)
ax.annotate('', xy=(G[0]+0.6, G[1]-0.4), xytext=(G[0]+0.65, G[1]-0.55),
            arrowprops=dict(arrowstyle='->', color=COLOR_ROJO, lw=2))
ax.text(G[0]+0.9, G[1]-0.9, '$M_i = -I_G\\alpha$', fontsize=12,
        color=COLOR_ROJO, fontweight='bold')

# Peso
draw_force_arrow(ax, G, (G[0], G[1]-1.5), color=COLOR_MORADO)
ax.text(G[0]+0.15, G[1]-1.7, '$m\\vec{g}$', fontsize=12, color=COLOR_MORADO)

# Leyenda
legend_elements = [
    Line2D([0], [0], color=COLOR_PUNTO, lw=2, label='Reacciones'),
    Line2D([0], [0], color=COLOR_ROJO, lw=2, label='Fuerzas de inercia'),
    Line2D([0], [0], color=COLOR_MORADO, lw=2, label='Peso'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

## 4. Reacciones en pares cinematicos

Las reacciones dependen del tipo de par:

| Par cinematico | Incognitas de reaccion | GDL restringidos |
|:---------------|:-----------------------|:-----------------|
| **Rotacion** (R) | $R_x, R_y$ (2 componentes de fuerza) | 2 |
| **Prismatico** (P) | $N$ (normal) + $M$ (momento) | 2 |
| **Leva** (C) | $N$ (normal, sin friccion) | 1 |
| **Rueda sin deslizamiento** (RSD) | $N + T$ (normal + tangencial) | 2 |

**Nota:** En el par prismatico, la reaccion se puede expresar equivalentemente como dos fuerzas
 normales $N$ y $N'$ separadas una distancia $d$, en lugar de $N + M$.

In [ ]:
# ── Reacciones en cada tipo de par ──
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

# --- Par de rotacion ---
ax = axes[0]
clean_ax(ax, 'Par de rotacion (R)')
ax.set_xlim(-2, 3)
ax.set_ylim(-2, 3)
# Eje
draw_joint(ax, (0, 0))
draw_ground(ax, (0, 0), 1.5, y_offset=-0.3)
# Barra
draw_link(ax, (0, 0), (2, 1.5), color=COLOR_PRINCIPAL, lw=5)
# Reacciones
draw_force_arrow(ax, (0, 0), (1.5, 0), color=COLOR_ROJO)
ax.text(1.6, 0.1, '$R_x$', fontsize=13, color=COLOR_ROJO, fontweight='bold')
draw_force_arrow(ax, (0, 0), (0, 1.5), color=COLOR_ROJO)
ax.text(0.15, 1.6, '$R_y$', fontsize=13, color=COLOR_ROJO, fontweight='bold')
ax.text(0, -1.5, '2 incognitas', fontsize=10, ha='center', style='italic')

# --- Par prismatico ---
ax = axes[1]
clean_ax(ax, 'Par prismatico (P)')
ax.set_xlim(-2, 4)
ax.set_ylim(-2, 3)
# Guia
ax.plot([-1, 3.5], [-0.4, -0.4], '-', color=COLOR_FIJO, lw=2)
ax.plot([-1, 3.5], [0.4, 0.4], '-', color=COLOR_FIJO, lw=2)
# Corredera
rect = Rectangle((0.5, -0.3), 1.5, 0.6, facecolor=COLOR_CLARO,
                  edgecolor=COLOR_PRINCIPAL, lw=2)
ax.add_patch(rect)
# Reacciones
draw_force_arrow(ax, (1.25, 0.4), (1.25, 1.5), color=COLOR_ROJO)
ax.text(1.4, 1.6, '$N$', fontsize=13, color=COLOR_ROJO, fontweight='bold')
# Momento
arc = Arc((1.25, 0), 1.2, 1.2, angle=0, theta1=30, theta2=300, color=COLOR_ROJO, lw=2)
ax.add_patch(arc)
ax.text(2.0, -0.8, '$M$', fontsize=13, color=COLOR_ROJO, fontweight='bold')
ax.text(1.25, -1.5, '2 incognitas: $N, M$', fontsize=10, ha='center', style='italic')

# --- Par leva ---
ax = axes[2]
clean_ax(ax, 'Par leva (C)')
ax.set_xlim(-2, 3)
ax.set_ylim(-2, 3)
# Perfil leva
theta_leva = np.linspace(0, 2*np.pi, 100)
r_leva = 0.8 + 0.3*np.sin(2*theta_leva)
ax.plot(r_leva*np.cos(theta_leva), r_leva*np.sin(theta_leva),
        color=COLOR_PRINCIPAL, lw=2.5)
# Seguidor
ax.plot([0.9, 0.9], [0, 2.5], '-', color=COLOR_FIJO, lw=3)
ax.plot(0.9, 0, 'o', color='black', ms=6, zorder=5)
# Reaccion normal
draw_force_arrow(ax, (0.9, 0), (2.0, 0), color=COLOR_ROJO)
ax.text(2.1, 0.1, '$N$', fontsize=13, color=COLOR_ROJO, fontweight='bold')
ax.text(0.5, -1.5, '1 incognita: $N$', fontsize=10, ha='center', style='italic')

# --- Par RSD ---
ax = axes[3]
clean_ax(ax, 'Rueda sin desliz. (RSD)')
ax.set_xlim(-2, 3)
ax.set_ylim(-2, 3)
# Suelo
ax.plot([-1.5, 2.5], [-1, -1], '-', color=COLOR_FIJO, lw=2)
draw_ground(ax, (0.5, -1), 3.0, y_offset=-0.1)
# Rueda
circle = plt.Circle((0.5, 0), 1.0, fill=False, edgecolor=COLOR_PRINCIPAL, lw=2.5)
ax.add_patch(circle)
ax.plot(0.5, 0, '+', color=COLOR_PRINCIPAL, ms=12, mew=2)
# Normal
draw_force_arrow(ax, (0.5, -1), (0.5, -2), color=COLOR_ROJO)
ax.text(0.7, -2.1, '$N$', fontsize=13, color=COLOR_ROJO, fontweight='bold')
# Tangencial
draw_force_arrow(ax, (0.5, -1), (1.7, -1), color=COLOR_NARANJA)
ax.text(1.8, -1.1, '$T$', fontsize=13, color=COLOR_NARANJA, fontweight='bold')
ax.text(0.5, -2.6, '2 incognitas: $N, T$', fontsize=10, ha='center', style='italic')

plt.tight_layout()
plt.show()

## 5. Rozamiento seco (Amontons-Coulomb)

El modelo clasico de friccion seca distingue dos regimenes:

**Estatico** (cuerpo en reposo relativo):
$$F \le \mu_s \, N$$

La fuerza de friccion puede tomar cualquier valor hasta el limite $\mu_s N$.

**Dinamico** (cuerpo en movimiento relativo):
$$T = \mu_d \, N \quad \text{con} \quad \mu_d < \mu_s$$

La fuerza de friccion es constante y se opone al movimiento relativo.

**Valores tipicos:**

| Materiales | $\mu_s$ | $\mu_d$ |
|:-----------|:--------|:--------|
| Acero-Acero (seco) | 0.6 | 0.4 |
| Acero-Acero (lubricado) | 0.15 | 0.06 |
| Acero-Bronce | 0.35 | 0.2 |

## 6. Metodo vectorial: Principio de D'Alembert

El metodo consiste en:

1. **Dibujar el DCL** (Diagrama de Cuerpo Libre) de **cada eslabon** por separado
2. Incluir: reacciones en pares, fuerzas externas, peso, y **fuerzas/momentos de inercia**
3. Plantear **3 ecuaciones de equilibrio por eslabon** ($N$ eslabones moviles):

$$\sum F_x^{(k)} = 0, \quad \sum F_y^{(k)} = 0, \quad \sum M^{(k)} = 0 \quad k = 1, \ldots, N$$

4. Se obtiene un **sistema de $3N$ ecuaciones** con $3N$ incognitas (reacciones + par motor)

**Ventaja:** Se obtienen TODAS las reacciones internas

**Desventaja:** Sistema grande, muchas ecuaciones

In [ ]:
# ── DCL completo del mecanismo biela-manivela (slider-crank) ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Parametros geometricos
theta2 = np.radians(45)
L2, L3 = 1.5, 4.0  # manivela, biela
A = np.array([L2*np.cos(theta2), L2*np.sin(theta2)])
# Posicion B sobre eje x
theta3 = np.arcsin(-A[1] / L3)
B = np.array([A[0] + L3*np.cos(theta3), 0.0])
O = np.array([0.0, 0.0])
G2 = (O + A) / 2
G3 = (A + B) / 2

# --- Eslabon 2 (manivela) ---
ax = axes[0]
clean_ax(ax, 'Eslabon 2 (Manivela)')
ax.set_xlim(-2, 4)
ax.set_ylim(-2, 4)
draw_link(ax, O, A, color=COLOR_ROJO, lw=7)
draw_joint(ax, O, fixed=True)
draw_joint(ax, A)
ax.plot(*G2, 'D', color=COLOR_NARANJA, ms=8, zorder=6)
ax.text(O[0]-0.5, O[1]-0.3, '$O$', fontsize=13, fontweight='bold')
ax.text(A[0]+0.2, A[1]+0.2, '$A$', fontsize=13, fontweight='bold')
ax.text(G2[0]+0.2, G2[1]+0.3, '$G_2$', fontsize=11, color=COLOR_NARANJA)

# Par motor en O
arc = Arc(O, 1.2, 1.2, angle=0, theta1=20, theta2=330, color=COLOR_PUNTO, lw=2.5)
ax.add_patch(arc)
ax.text(O[0]+0.7, O[1]-0.9, '$M_{motor}$', fontsize=12, color=COLOR_PUNTO, fontweight='bold')

# Reacciones en O
draw_force_arrow(ax, O, (O[0]-1.2, O[1]), color=COLOR_PUNTO)
ax.text(O[0]-1.8, O[1]+0.1, '$R_{Ox}$', fontsize=11, color=COLOR_PUNTO)
draw_force_arrow(ax, O, (O[0], O[1]-1.2), color=COLOR_PUNTO)
ax.text(O[0]+0.15, O[1]-1.4, '$R_{Oy}$', fontsize=11, color=COLOR_PUNTO)

# Reacciones en A (de biela sobre manivela)
draw_force_arrow(ax, A, (A[0]+1.0, A[1]), color=COLOR_MORADO)
ax.text(A[0]+1.1, A[1]+0.1, '$R_{32x}$', fontsize=11, color=COLOR_MORADO)
draw_force_arrow(ax, A, (A[0], A[1]+1.0), color=COLOR_MORADO)
ax.text(A[0]+0.15, A[1]+1.1, '$R_{32y}$', fontsize=11, color=COLOR_MORADO)

# Inercia en G2
draw_force_arrow(ax, G2, (G2[0]-0.8, G2[1]-0.3), color=COLOR_ROJO)
ax.text(G2[0]-1.6, G2[1]-0.5, '$\\vec{F}_{i2}$', fontsize=11, color=COLOR_ROJO)

# Peso
draw_force_arrow(ax, G2, (G2[0], G2[1]-0.8), color=COLOR_FIJO)
ax.text(G2[0]+0.15, G2[1]-1.0, '$m_2 g$', fontsize=10, color=COLOR_FIJO)

# --- Eslabon 3 (biela) ---
ax = axes[1]
clean_ax(ax, 'Eslabon 3 (Biela)')
ax.set_xlim(-1.5, 7)
ax.set_ylim(-3, 3)
draw_link(ax, A, B, color=COLOR_PRINCIPAL, lw=7)
draw_joint(ax, A)
draw_joint(ax, B)
ax.plot(*G3, 'D', color=COLOR_NARANJA, ms=8, zorder=6)
ax.text(A[0]-0.5, A[1]+0.2, '$A$', fontsize=13, fontweight='bold')
ax.text(B[0]+0.2, B[1]+0.3, '$B$', fontsize=13, fontweight='bold')
ax.text(G3[0]+0.2, G3[1]+0.3, '$G_3$', fontsize=11, color=COLOR_NARANJA)

# Reacciones en A (de manivela sobre biela: -R_32)
draw_force_arrow(ax, A, (A[0]-1.0, A[1]), color=COLOR_MORADO)
ax.text(A[0]-1.8, A[1]+0.1, '$R_{23x}$', fontsize=11, color=COLOR_MORADO)
draw_force_arrow(ax, A, (A[0], A[1]-1.0), color=COLOR_MORADO)
ax.text(A[0]+0.15, A[1]-1.3, '$R_{23y}$', fontsize=11, color=COLOR_MORADO)

# Reacciones en B (de corredera sobre biela)
draw_force_arrow(ax, B, (B[0]+1.0, B[1]), color=COLOR_MORADO)
ax.text(B[0]+1.1, B[1]+0.1, '$R_{43x}$', fontsize=11, color=COLOR_MORADO)
draw_force_arrow(ax, B, (B[0], B[1]+1.0), color=COLOR_MORADO)
ax.text(B[0]+0.15, B[1]+1.1, '$R_{43y}$', fontsize=11, color=COLOR_MORADO)

# Inercia en G3
draw_force_arrow(ax, G3, (G3[0]-1.0, G3[1]+0.5), color=COLOR_ROJO)
ax.text(G3[0]-1.8, G3[1]+0.6, '$\\vec{F}_{i3}$', fontsize=11, color=COLOR_ROJO)

# Peso
draw_force_arrow(ax, G3, (G3[0], G3[1]-1.0), color=COLOR_FIJO)
ax.text(G3[0]+0.15, G3[1]-1.3, '$m_3 g$', fontsize=10, color=COLOR_FIJO)

# --- Eslabon 4 (corredera) ---
ax = axes[2]
clean_ax(ax, 'Eslabon 4 (Corredera)')
ax.set_xlim(1, 8)
ax.set_ylim(-3, 3)
# Guia
ax.plot([2, 7], [-0.5, -0.5], '-', color=COLOR_FIJO, lw=2)
ax.plot([2, 7], [0.5, 0.5], '-', color=COLOR_FIJO, lw=2)
# Corredera
rect = Rectangle((B[0]-0.6, -0.4), 1.2, 0.8, facecolor=COLOR_CLARO,
                  edgecolor=COLOR_PRINCIPAL, lw=2)
ax.add_patch(rect)
ax.text(B[0], 0.8, '$B$', fontsize=13, fontweight='bold', ha='center')

# Reaccion de biela sobre corredera
draw_force_arrow(ax, B, (B[0]-1.0, B[1]), color=COLOR_MORADO)
ax.text(B[0]-1.8, B[1]+0.1, '$R_{34x}$', fontsize=11, color=COLOR_MORADO)
draw_force_arrow(ax, B, (B[0], B[1]-1.0), color=COLOR_MORADO)
ax.text(B[0]+0.15, B[1]-1.3, '$R_{34y}$', fontsize=11, color=COLOR_MORADO)

# Normal del suelo
draw_force_arrow(ax, (B[0], -0.5), (B[0], -1.8), color=COLOR_ROJO)
ax.text(B[0]+0.2, -2.0, '$N_{14}$', fontsize=11, color=COLOR_ROJO)

# Fuerza externa (resistente)
draw_force_arrow(ax, (B[0]+0.6, 0), (B[0]+1.8, 0), color=COLOR_PUNTO)
ax.text(B[0]+1.9, 0.1, '$F_{ext}$', fontsize=11, color=COLOR_PUNTO, fontweight='bold')

# Inercia
draw_force_arrow(ax, B, (B[0]-0.8, B[1]+0.8), color=COLOR_ROJO)
ax.text(B[0]-1.5, B[1]+0.9, '$\\vec{F}_{i4}$', fontsize=11, color=COLOR_ROJO)

# Peso
draw_force_arrow(ax, B, (B[0], B[1]-0.6), color=COLOR_FIJO)
ax.text(B[0]+0.15, B[1]-0.9, '$m_4 g$', fontsize=10, color=COLOR_FIJO)

plt.tight_layout()
plt.show()

## 7. Metodo analitico: Principio de Potencias Virtuales

En lugar de plantear el equilibrio de cada eslabon por separado, se puede aplicar
 el balance de potencias a **todo el mecanismo**:

$$\sum P = P_{\text{motriz}} + P_{\text{grav}} + P_{\text{inercia}} + P_{\text{pasivas}} = 0$$

donde:
- $P = \vec{F} \cdot \vec{v}$ para fuerzas
- $P = M \cdot \omega$ para pares

**Ventaja:** Las reacciones internas no aparecen (potencia nula), por lo que se obtiene
 directamente el par motor con **una sola ecuacion**.

**Desventaja:** No proporciona las reacciones internas (si se necesitan, hay que usar D'Alembert).

Para el mecanismo biela-manivela:

$$M_{\text{motor}} \cdot \omega_2 + (-F_{\text{ext}}) \cdot v_B + \sum_k (-m_k g \cdot v_{Gk,y})
 + \sum_k (-m_k \vec{a}_{Gk} \cdot \vec{v}_{Gk} - I_{Gk} \alpha_k \omega_k) = 0$$

## 8. Analisis dinamico inverso

**Problema:** Dado el movimiento (posicion, velocidad, aceleracion), determinar las fuerzas/pares
 necesarios para producirlo.

**Procedimiento:**
1. Resolver la **cinematica** completa: posiciones, velocidades, aceleraciones de todos los eslabones
2. Calcular las **fuerzas de inercia** ($F_i = -ma_G$, $M_i = -I_G \alpha$)
3. Plantear el sistema de ecuaciones de equilibrio (D'Alembert) o la ecuacion de potencias
4. Resolver el sistema lineal para obtener el par motor y las reacciones

El resultado tipico es **$M_{\text{motor}}(\theta)$**: el par motor necesario en funcion del angulo de entrada.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  ANALISIS DINAMICO INVERSO COMPLETO: Mecanismo biela-manivela
#  Paso 1: Cinematica (fsolve + Jacobiano, patron del notebook 02)
#  Paso 2: D'Alembert -> par motor y reacciones
# ══════════════════════════════════════════════════════════════════════════

# ── Parametros del mecanismo ──
L2 = 0.10   # m - longitud manivela
L3 = 0.30   # m - longitud biela
m2 = 0.5    # kg - masa manivela
m3 = 1.0    # kg - masa biela
m4 = 2.0    # kg - masa corredera (piston)
IG2 = m2 * L2**2 / 12   # momento de inercia manivela (barra delgada)
IG3 = m3 * L3**2 / 12   # momento de inercia biela
g = 9.81    # m/s^2
omega2 = 300 * 2 * np.pi / 60  # rad/s (300 rpm)
alpha2 = 0.0                    # rad/s^2 (velocidad constante)
F_ext = 500.0  # N - fuerza resistente sobre piston (gas)

In [ ]:
# ── Ecuaciones de cierre del lazo (cinematica de posiciones) ──
# Lazo vectorial: O + L2*e^(j*theta2) + L3*e^(j*theta3) - xB*e_x = 0
#   eq1: L2*cos(theta2) + L3*cos(theta3) - xB = 0
#   eq2: L2*sin(theta2) + L3*sin(theta3) = 0

def pos_equations(q, theta2):
    """Ecuaciones de cierre para posicion.
    q = [theta3, xB]
    """
    theta3, xB = q
    eq1 = L2 * np.cos(theta2) + L3 * np.cos(theta3) - xB
    eq2 = L2 * np.sin(theta2) + L3 * np.sin(theta3)
    return [eq1, eq2]

def pos_jacobian(q, theta2):
    """Jacobiano de las ecuaciones de posicion respecto a q = [theta3, xB]."""
    theta3, xB = q
    return np.array([
        [-L3 * np.sin(theta3), -1.0],
        [ L3 * np.cos(theta3),  0.0]
    ])

def solve_kinematics(theta2_val, q0):
    """Resuelve posicion, velocidad y aceleracion para un angulo theta2."""
    # ── Posicion ──
    q_sol = fsolve(pos_equations, q0, args=(theta2_val,), fprime=pos_jacobian)
    theta3_val, xB_val = q_sol
    
    # ── Velocidad (derivando las ecuaciones de cierre) ──
    # J * dq/dt = -b * omega2
    J = pos_jacobian(q_sol, theta2_val)
    b = np.array([
        -L2 * np.sin(theta2_val),
         L2 * np.cos(theta2_val)
    ])
    dqdt = np.linalg.solve(J, -b * omega2)
    omega3 = dqdt[0]
    vB = dqdt[1]
    
    # ── Aceleracion (derivando las ecuaciones de velocidad) ──
    # J * d2q/dt2 = -c
    c = np.array([
        -L2 * np.cos(theta2_val) * omega2**2 - L3 * np.cos(theta3_val) * omega3**2
            + L2 * np.sin(theta2_val) * alpha2,
        -L2 * np.sin(theta2_val) * omega2**2 - L3 * np.sin(theta3_val) * omega3**2
            - L2 * np.cos(theta2_val) * alpha2
    ])
    d2qdt2 = np.linalg.solve(J, -c)
    alpha3 = d2qdt2[0]
    aB = d2qdt2[1]
    
    return theta3_val, xB_val, omega3, vB, alpha3, aB

# ── Resolver para todo un ciclo ──
N_pts = 360
theta2_arr = np.linspace(0, 2*np.pi, N_pts, endpoint=False)

theta3_arr = np.zeros(N_pts)
xB_arr = np.zeros(N_pts)
omega3_arr = np.zeros(N_pts)
vB_arr = np.zeros(N_pts)
alpha3_arr = np.zeros(N_pts)
aB_arr = np.zeros(N_pts)

q0 = [0.0, L2 + L3]  # estimacion inicial
for i, th2 in enumerate(theta2_arr):
    res = solve_kinematics(th2, q0)
    theta3_arr[i], xB_arr[i] = res[0], res[1]
    omega3_arr[i], vB_arr[i] = res[2], res[3]
    alpha3_arr[i], aB_arr[i] = res[4], res[5]
    q0 = [res[0], res[1]]  # continuacion

In [ ]:
# ── Aceleraciones de los centros de masas ──
# G2 = punto medio de la manivela OA
# G3 = punto medio de la biela AB
# G4 = piston (punto B)

# Posiciones de los centros de masas
xG2 = (L2/2) * np.cos(theta2_arr)
yG2 = (L2/2) * np.sin(theta2_arr)
xG3 = L2 * np.cos(theta2_arr) + (L3/2) * np.cos(theta3_arr)
yG3 = L2 * np.sin(theta2_arr) + (L3/2) * np.sin(theta3_arr)

# Aceleraciones del centro de masas de la manivela (G2)
# a_G2 = d/dt(v_G2), con v_G2 = omega2 x OG2
aG2x = -(L2/2) * (omega2**2 * np.cos(theta2_arr) + alpha2 * np.sin(theta2_arr))
aG2y = -(L2/2) * (omega2**2 * np.sin(theta2_arr) - alpha2 * np.cos(theta2_arr))

# Aceleraciones del centro de masas de la biela (G3)
# a_G3 = a_A + alpha3 x AG3 - omega3^2 * AG3
aAx = -L2 * (omega2**2 * np.cos(theta2_arr) + alpha2 * np.sin(theta2_arr))
aAy = -L2 * (omega2**2 * np.sin(theta2_arr) - alpha2 * np.cos(theta2_arr))

aG3x = aAx - (L3/2) * (alpha3_arr * np.sin(theta3_arr) + omega3_arr**2 * np.cos(theta3_arr))
aG3y = aAy + (L3/2) * (alpha3_arr * np.cos(theta3_arr) - omega3_arr**2 * np.sin(theta3_arr))

# Aceleracion del piston (G4 = B)
aG4x = aB_arr  # solo componente x
aG4y = np.zeros(N_pts)  # se mueve solo en x

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  DINAMICA INVERSA: D'Alembert -> sistema de ecuaciones lineales
#  Para cada posicion theta2, resolver A*x = b
# ══════════════════════════════════════════════════════════════════════════

M_motor_dalembert = np.zeros(N_pts)
ROx_arr = np.zeros(N_pts)
ROy_arr = np.zeros(N_pts)
R32x_arr = np.zeros(N_pts)
R32y_arr = np.zeros(N_pts)
R43x_arr = np.zeros(N_pts)
R43y_arr = np.zeros(N_pts)
N14_arr = np.zeros(N_pts)

for i in range(N_pts):
    th2 = theta2_arr[i]
    th3 = theta3_arr[i]
    
    # Posiciones de articulaciones
    Ax = L2 * np.cos(th2)
    Ay = L2 * np.sin(th2)
    Bx = xB_arr[i]
    By = 0.0
    
    # ── Sistema 3*3 eslabones = 9 ecuaciones, 8 incognitas + M_motor ──
    # Incognitas: [ROx, ROy, R32x, R32y, R43x, R43y, N14, M_motor]
    # x = [0:ROx, 1:ROy, 2:R32x, 3:R32y, 4:R43x, 5:R43y, 6:N14, 7:M_motor]
    
    A_mat = np.zeros((8, 8))
    b_vec = np.zeros(8)
    
    # ── Eslabon 2 (manivela): sum Fx=0, sum Fy=0, sum M_O=0 ──
    # sum Fx: ROx + R32x = m2*aG2x  (inercia al lado derecho)
    A_mat[0, 0] = 1.0  # ROx
    A_mat[0, 2] = 1.0  # R32x
    b_vec[0] = m2 * aG2x[i]
    
    # sum Fy: ROy + R32y - m2*g = m2*aG2y
    A_mat[1, 1] = 1.0  # ROy
    A_mat[1, 3] = 1.0  # R32y
    b_vec[1] = m2 * aG2y[i] + m2 * g
    
    # sum M_O: M_motor + R32x*Ay - R32y*Ax + m2*g*(L2/2)*cos(th2) =
    #          m2*(aG2x*yG2 - aG2y*xG2) + IG2*alpha3  (wait: alpha2 for link 2)
    # Momento de inercia respecto a O para eslabon 2:
    # Using direct D'Alembert: sum M_O = I_G2*alpha2 + m2*(xG2*aG2y - yG2*aG2x)
    #   (cross product r x ma for translational inertia)
    A_mat[2, 2] =  Ay   # R32x * Ay
    A_mat[2, 3] = -Ax   # -R32y * Ax
    A_mat[2, 7] =  1.0  # M_motor
    xg2 = xG2[i]
    yg2 = yG2[i]
    b_vec[2] = (IG2 * alpha2 + m2 * (xg2 * aG2y[i] - yg2 * aG2x[i])
               + m2 * g * xg2)
    
    # ── Eslabon 3 (biela): sum Fx=0, sum Fy=0, sum M_A=0 ──
    # Nota: R23 = -R32 (accion-reaccion), pero escribimos directamente
    # sum Fx: -R32x + R43x = m3*aG3x
    A_mat[3, 2] = -1.0  # -R32x (reaccion en A sobre biela)
    A_mat[3, 4] =  1.0  # R43x
    b_vec[3] = m3 * aG3x[i]
    
    # sum Fy: -R32y + R43y - m3*g = m3*aG3y
    A_mat[4, 3] = -1.0  # -R32y
    A_mat[4, 5] =  1.0  # R43y
    b_vec[4] = m3 * aG3y[i] + m3 * g
    
    # sum M_A: R43x*(By-Ay) - R43y*(Bx-Ax) + m3*g*((L3/2)*cos(th3)) =
    #          IG3*alpha3 + m3*(rG3_from_A x aG3)
    rG3Ax = xG3[i] - Ax  # vector AG3
    rG3Ay = yG3[i] - Ay
    rBAx = Bx - Ax
    rBAy = By - Ay
    A_mat[5, 4] =  rBAy   # R43x * (By-Ay)
    A_mat[5, 5] = -rBAx   # -R43y * (Bx-Ax)
    b_vec[5] = (IG3 * alpha3_arr[i]
               + m3 * (rG3Ax * aG3y[i] - rG3Ay * aG3x[i])
               + m3 * g * rG3Ax)
    
    # ── Eslabon 4 (corredera): sum Fx=0, sum Fy=0 ──
    # (solo 2 ecuaciones utiles, no tiene rotacion significativa)
    # sum Fx: -R43x - F_ext = m4*aG4x
    A_mat[6, 4] = -1.0  # -R43x
    b_vec[6] = m4 * aG4x[i] + F_ext
    
    # sum Fy: -R43y + N14 - m4*g = 0  (aG4y = 0)
    A_mat[7, 5] = -1.0  # -R43y
    A_mat[7, 6] =  1.0  # N14
    b_vec[7] = m4 * g
    
    # ── Resolver ──
    x_sol = np.linalg.solve(A_mat, b_vec)
    ROx_arr[i] = x_sol[0]
    ROy_arr[i] = x_sol[1]
    R32x_arr[i] = x_sol[2]
    R32y_arr[i] = x_sol[3]
    R43x_arr[i] = x_sol[4]
    R43y_arr[i] = x_sol[5]
    N14_arr[i] = x_sol[6]
    M_motor_dalembert[i] = x_sol[7]

In [ ]:
# ── Grafica: Par motor vs angulo de entrada ──
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

theta2_deg = np.degrees(theta2_arr)

# Par motor
ax = axes[0, 0]
ax.plot(theta2_deg, M_motor_dalembert, color=COLOR_ROJO, lw=2)
ax.axhline(y=0, color='gray', ls='--', lw=0.8)
ax.axhline(y=M_motor_dalembert.mean(), color=COLOR_PUNTO, ls='--', lw=1.5,
           label=f'Media = {M_motor_dalembert.mean():.2f} Nm')
ax.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax.set_ylabel(r'$M_{motor}$ (Nm)', fontsize=12)
ax.set_title('Par motor (D\'Alembert)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Reacciones en O
ax = axes[0, 1]
ax.plot(theta2_deg, ROx_arr, color=COLOR_PRINCIPAL, lw=1.5, label='$R_{Ox}$')
ax.plot(theta2_deg, ROy_arr, color=COLOR_NARANJA, lw=1.5, label='$R_{Oy}$')
RO_mag = np.sqrt(ROx_arr**2 + ROy_arr**2)
ax.plot(theta2_deg, RO_mag, color=COLOR_ROJO, lw=2, label='$|R_O|$')
ax.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax.set_ylabel('Fuerza (N)', fontsize=12)
ax.set_title('Reacciones en O (apoyo)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Posicion y velocidad del piston
ax = axes[1, 0]
ax.plot(theta2_deg, xB_arr * 1000, color=COLOR_PRINCIPAL, lw=2, label='$x_B$ (mm)')
ax2 = ax.twinx()
ax2.plot(theta2_deg, vB_arr, color=COLOR_NARANJA, lw=1.5, ls='--', label='$v_B$ (m/s)')
ax.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax.set_ylabel('Posicion $x_B$ (mm)', fontsize=12, color=COLOR_PRINCIPAL)
ax2.set_ylabel('Velocidad $v_B$ (m/s)', fontsize=12, color=COLOR_NARANJA)
ax.set_title('Cinematica del piston', fontsize=13, fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=10)
ax.grid(True, alpha=0.3)

# Normal en guia
ax = axes[1, 1]
ax.plot(theta2_deg, N14_arr, color=COLOR_MORADO, lw=2)
ax.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax.set_ylabel('$N_{14}$ (N)', fontsize=12)
ax.set_title('Normal piston-guia', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  VERIFICACION: Potencias virtuales (debe dar el mismo M_motor)
# ══════════════════════════════════════════════════════════════════════════

M_motor_power = np.zeros(N_pts)

for i in range(N_pts):
    th2 = theta2_arr[i]
    th3 = theta3_arr[i]
    om3 = omega3_arr[i]
    
    # Velocidades de los centros de masas
    vG2x = -(L2/2) * omega2 * np.sin(th2)
    vG2y =  (L2/2) * omega2 * np.cos(th2)
    
    vAx = -L2 * omega2 * np.sin(th2)
    vAy =  L2 * omega2 * np.cos(th2)
    vG3x = vAx - (L3/2) * om3 * np.sin(th3)
    vG3y = vAy + (L3/2) * om3 * np.cos(th3)
    
    vG4x = vB_arr[i]
    vG4y = 0.0
    
    # Potencia de fuerzas externas
    P_ext = -F_ext * vG4x  # fuerza resistente sobre piston
    
    # Potencia de la gravedad
    P_grav = -m2 * g * vG2y - m3 * g * vG3y - m4 * g * vG4y
    
    # Potencia de fuerzas de inercia
    P_inercia = -(m2 * (aG2x[i]*vG2x + aG2y[i]*vG2y) + IG2 * alpha2 * omega2
                 + m3 * (aG3x[i]*vG3x + aG3y[i]*vG3y) + IG3 * alpha3_arr[i] * om3
                 + m4 * (aG4x[i]*vG4x))
    
    # Balance: P_motor + P_ext + P_grav + P_inercia = 0
    # P_motor = M_motor * omega2
    # M_motor = -(P_ext + P_grav + P_inercia) / omega2
    M_motor_power[i] = -(P_ext + P_grav + P_inercia) / omega2

# Comparacion
error = np.abs(M_motor_dalembert - M_motor_power)

In [ ]:
# ── Comparacion D'Alembert vs Potencias Virtuales ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[3, 1])

ax1.plot(theta2_deg, M_motor_dalembert, color=COLOR_ROJO, lw=2.5,
         label="D'Alembert (vectorial)")
ax1.plot(theta2_deg, M_motor_power, color=COLOR_PUNTO, lw=1.5, ls='--',
         label='Potencias virtuales')
ax1.axhline(y=0, color='gray', ls='--', lw=0.8)
ax1.set_ylabel(r'$M_{motor}$ (Nm)', fontsize=12)
ax1.set_title('Comparacion: D\'Alembert vs Potencias Virtuales', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.semilogy(theta2_deg, error + 1e-16, color=COLOR_MORADO, lw=1.5)
ax2.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax2.set_ylabel('Error (Nm)', fontsize=12)
ax2.set_title('Error absoluto entre metodos', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Analisis dinamico directo

**Problema:** Dadas las fuerzas aplicadas (par motor, cargas, etc.), determinar el movimiento del mecanismo.

**Procedimiento:**
1. Reducir el mecanismo a un **sistema de 1 GDL** (coordenada generalizada $\theta_2$)
2. Plantear la ecuacion de movimiento reducida:

$$I_{eq}(\theta_2)\,\ddot{\theta}_2 = M_{\text{motor}} - M_{\text{resistente}}(\theta_2, \dot{\theta}_2)$$

donde $I_{eq}$ es el **momento de inercia equivalente reducido** al eje de entrada:

$$I_{eq} = \sum_k \left( m_k \left(\frac{v_{Gk}}{\omega_2}\right)^2 + I_{Gk} \left(\frac{\omega_k}{\omega_2}\right)^2 \right)$$

3. Integrar la ODE resultante con un metodo numerico (e.g., `solve_ivp`)

**Nota:** En el analisis directo, $\omega_2$ ya no es constante; varia con el tiempo.

## 10. Teorema de las fuerzas vivas

El **Teorema de las Fuerzas Vivas** (o Teorema de la Energia Cinetica) establece que:

$$W_{\text{total}} = \Delta T = T_f - T_i$$

donde $T$ es la energia cinetica total del sistema:

$$T = \sum_k \left( \frac{1}{2} m_k v_{Gk}^2 + \frac{1}{2} I_{Gk} \omega_k^2 \right)$$

**Aplicaciones:**
- Calcular la velocidad en una posicion dada conociendo el trabajo realizado
- Dimensionar volantes de inercia (la variacion de energia cinetica se almacena en el volante)
- Verificar resultados de analisis inverso/directo

Para el mecanismo con inercia equivalente:

$$W_{\text{total}} = \frac{1}{2} I_{eq} \omega_2^2 \Big|_{\theta_i}^{\theta_f}$$

In [ ]:
# ── Verificacion energetica: Teorema de las Fuerzas Vivas ──

# Energia cinetica de cada eslabon en cada posicion
# T = 1/2 * m * vG^2 + 1/2 * IG * omega^2

# Velocidades de los centros de masas
vG2x_arr = -(L2/2) * omega2 * np.sin(theta2_arr)
vG2y_arr =  (L2/2) * omega2 * np.cos(theta2_arr)
vG2_sq = vG2x_arr**2 + vG2y_arr**2

vAx_arr = -L2 * omega2 * np.sin(theta2_arr)
vAy_arr =  L2 * omega2 * np.cos(theta2_arr)
vG3x_arr = vAx_arr - (L3/2) * omega3_arr * np.sin(theta3_arr)
vG3y_arr = vAy_arr + (L3/2) * omega3_arr * np.cos(theta3_arr)
vG3_sq = vG3x_arr**2 + vG3y_arr**2

vG4_sq = vB_arr**2  # piston

T_arr = (0.5 * m2 * vG2_sq + 0.5 * IG2 * omega2**2
       + 0.5 * m3 * vG3_sq + 0.5 * IG3 * omega3_arr**2
       + 0.5 * m4 * vG4_sq)

# Trabajo del par motor (integral acumulativa)
dtheta = theta2_arr[1] - theta2_arr[0]
W_motor = np.cumsum(M_motor_dalembert * dtheta)

# Trabajo de la fuerza externa
dxB = np.gradient(xB_arr, theta2_arr) * dtheta  # incrementos
W_ext = np.cumsum(-F_ext * np.gradient(xB_arr, theta2_arr) * dtheta)

# Trabajo de la gravedad
W_grav = np.cumsum(-m2*g*np.gradient(yG2, theta2_arr)*dtheta
                   -m3*g*np.gradient(yG3, theta2_arr)*dtheta)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(theta2_deg, T_arr, color=COLOR_ROJO, lw=2.5, label='Energia cinetica $T$')
ax.plot(theta2_deg, T_arr[0] + W_motor + W_ext + W_grav, color=COLOR_PUNTO,
        lw=1.5, ls='--', label='$T_0 + W_{total}$ (verificacion)')
ax.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax.set_ylabel('Energia (J)', fontsize=12)
ax.set_title('Verificacion: Teorema de las Fuerzas Vivas', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Equilibrado de rotores

Un rotor esta **desequilibrado** cuando su centro de masas no coincide con el eje de giro,
 lo que produce fuerzas centrifugas que generan vibraciones.

### Equilibrado estatico (1 plano)

Se anaden masas de correccion en un **unico plano** para que:

$$\sum_k m_k \, r_k \, e^{j\theta_k} = 0$$

Equivalente a: $\sum m_k r_k \cos\theta_k = 0$ y $\sum m_k r_k \sin\theta_k = 0$.

### Equilibrado dinamico (2 planos)

Ademas del equilibrio de fuerzas, se requiere equilibrio de **momentos**:

$$\sum_k m_k \, r_k \, e^{j\theta_k} = 0 \quad \text{(fuerzas)}$$

$$\sum_k m_k \, r_k \, z_k \, e^{j\theta_k} = 0 \quad \text{(momentos)}$$

donde $z_k$ es la posicion axial de cada masa. Se necesitan **2 masas de correccion** en 2 planos.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  EQUILIBRADO DINAMICO DE UN ROTOR
# ══════════════════════════════════════════════════════════════════════════

# Masas desequilibradas (conocidas)
# Formato: (m*r [kg*m], theta [grados], z [m])
masas = [
    (0.005, 30,  0.1),   # masa 1
    (0.008, 150, 0.3),   # masa 2
    (0.003, 270, 0.5),   # masa 3
]

# Planos de correccion
zA = 0.0   # plano A (izquierdo)
zB = 0.6   # plano B (derecho)
rA = 0.05  # radio de correccion plano A
rB = 0.05  # radio de correccion plano B

for i, (mr, theta, z) in enumerate(masas):

# ── Ecuaciones de equilibrado ──
# Fuerzas: sum(mr_i * e^(j*theta_i)) + mA*rA*e^(j*thetaA) + mB*rB*e^(j*thetaB) = 0
# Momentos respecto a A: sum(mr_i * (z_i-zA) * e^(j*theta_i)) + mB*rB*(zB-zA)*e^(j*thetaB) = 0

# Paso 1: Resolver plano B a partir del equilibrio de momentos respecto a A
sum_moments = sum(mr * (z - zA) * np.exp(1j * np.radians(theta))
                  for mr, theta, z in masas)
mB_rB_complex = -sum_moments / (zB - zA)
mB_rB = np.abs(mB_rB_complex)
thetaB = np.degrees(np.angle(mB_rB_complex)) % 360
mB = mB_rB / rB

# Paso 2: Resolver plano A a partir del equilibrio de fuerzas
sum_forces = sum(mr * np.exp(1j * np.radians(theta))
                 for mr, theta, z in masas)
mA_rA_complex = -(sum_forces + mB_rB_complex)
mA_rA = np.abs(mA_rA_complex)
thetaA = np.degrees(np.angle(mA_rA_complex)) % 360
mA = mA_rA / rA


# ── Verificacion ──
all_masas = masas + [(mA_rA, thetaA, zA), (mB_rB, thetaB, zB)]
check_f = sum(mr * np.exp(1j * np.radians(theta))
              for mr, theta, z in all_masas)
check_m = sum(mr * z * np.exp(1j * np.radians(theta))
              for mr, theta, z in all_masas)

In [ ]:
# ── Diagrama del equilibrado del rotor ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Vista frontal (plano xy) ---
ax = ax1
clean_ax(ax, 'Vista frontal - Vectores m*r')
ax.set_xlim(-0.015, 0.015)
ax.set_ylim(-0.015, 0.015)

# Eje
ax.plot(0, 0, '+', color='black', ms=15, mew=2, zorder=5)
circle = plt.Circle((0, 0), 0.012, fill=False, edgecolor=COLOR_FIJO, ls='--', lw=1)
ax.add_patch(circle)

colors_masas = [COLOR_ROJO, COLOR_NARANJA, COLOR_MORADO]
for i, (mr, theta, z) in enumerate(masas):
    rad = np.radians(theta)
    ax.annotate('', xy=(mr*np.cos(rad), mr*np.sin(rad)),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors_masas[i], lw=2.5))
    ax.text(mr*np.cos(rad)*1.15, mr*np.sin(rad)*1.15,
            f'$m_{i+1}r_{i+1}$\n{mr*1000:.0f} g$\\cdot$mm\n{theta}$^\\circ$',
            fontsize=9, ha='center', color=colors_masas[i])

# Masas de correccion
for label, mr, theta, col in [('A', mA_rA, thetaA, COLOR_PUNTO),
                               ('B', mB_rB, thetaB, COLOR_PRINCIPAL)]:
    rad = np.radians(theta)
    ax.annotate('', xy=(mr*np.cos(rad), mr*np.sin(rad)),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=3))
    ax.text(mr*np.cos(rad)*1.2, mr*np.sin(rad)*1.2,
            f'Corr. {label}\n{mr*1000:.2f} g$\\cdot$mm\n{theta:.1f}$^\\circ$',
            fontsize=9, ha='center', color=col, fontweight='bold')

ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
ax.set_xlabel('x (m)', fontsize=11)
ax.set_ylabel('y (m)', fontsize=11)

# --- Vista lateral (eje z) ---
ax = ax2
clean_ax(ax, 'Vista lateral - Posicion axial')
ax.set_xlim(-0.1, 0.8)
ax.set_ylim(-0.5, 0.5)

# Eje del rotor
ax.plot([-0.05, 0.65], [0, 0], '-', color=COLOR_FIJO, lw=3)

# Masas originales
for i, (mr, theta, z) in enumerate(masas):
    ax.plot(z, 0.15, 'o', color=colors_masas[i], ms=12, zorder=5)
    ax.text(z, 0.25, f'$m_{i+1}$\nz={z}', fontsize=9, ha='center', color=colors_masas[i])

# Planos de correccion
for label, z, col in [('A', zA, COLOR_PUNTO), ('B', zB, COLOR_PRINCIPAL)]:
    ax.axvline(z, color=col, ls='--', lw=1.5, alpha=0.5)
    ax.plot(z, -0.15, 's', color=col, ms=12, zorder=5)
    ax.text(z, -0.3, f'Plano {label}\nz={z}', fontsize=9, ha='center',
            color=col, fontweight='bold')

ax.set_xlabel('z (m)', fontsize=11)

plt.tight_layout()
plt.show()

## 12. Ejercicios resueltos

---

### Ejercicio 1: Ecuaciones de D'Alembert para una barra en rotacion

Una barra homogenea de masa $m = 2$ kg y longitud $L = 0.5$ m gira alrededor
 de un extremo $O$ fijo. En un instante dado, $\theta = 60^\circ$, $\omega = 10$ rad/s, $\alpha = 5$ rad/s$^2$.

Calcular las reacciones en $O$ y el par motor necesario.

**Desarrollo:**

**Momento de inercia respecto a $O$ (Steiner):**

$$I_O = I_G + m\left(\frac{L}{2}\right)^2 = \frac{mL^2}{12} + \frac{mL^2}{4} = \frac{mL^2}{3}$$

$$I_O = \frac{2 \times 0.5^2}{3} = 0.1667 \text{ kg·m}^2$$

**Aceleracion del centro de masas** ($G$ en el punto medio):

$$a_{Gx} = -\frac{L}{2}\left(\omega^2 \cos\theta + \alpha \sin\theta\right) = -14.17 \text{ m/s}^2$$

$$a_{Gy} = -\frac{L}{2}\left(\omega^2 \sin\theta - \alpha \cos\theta\right) = -21.00 \text{ m/s}^2$$

**Ecuaciones de D'Alembert (equilibrio en $O$):**

$$\sum F_x: \quad R_{Ox} = m\,a_{Gx} = -28.35 \text{ N}$$

$$\sum F_y: \quad R_{Oy} = m\,a_{Gy} + m\,g = -22.38 \text{ N}$$

$$|R_O| = \sqrt{R_{Ox}^2 + R_{Oy}^2} = 36.12 \text{ N}$$

$$\sum M_O: \quad M_{motor} = I_O\,\alpha + m\,g\,\frac{L}{2}\cos\theta$$

$$\boxed{M_{motor} = 5.74 \text{ Nm}}$$

In [ ]:
# ── Ejercicio 1: Barra en rotacion pura ──
m_ej1 = 2.0     # kg
L_ej1 = 0.5     # m
theta_ej1 = np.radians(60)  # rad
omega_ej1 = 10.0   # rad/s
alpha_ej1 = 5.0    # rad/s^2
IG_ej1 = m_ej1 * L_ej1**2 / 12
IO_ej1 = m_ej1 * L_ej1**2 / 3  # Steiner: IG + m*(L/2)^2

# Centro de masas
xG = (L_ej1/2) * np.cos(theta_ej1)
yG = (L_ej1/2) * np.sin(theta_ej1)

# Aceleracion del centro de masas
aGx = -(L_ej1/2) * (omega_ej1**2 * np.cos(theta_ej1) + alpha_ej1 * np.sin(theta_ej1))
aGy = -(L_ej1/2) * (omega_ej1**2 * np.sin(theta_ej1) - alpha_ej1 * np.cos(theta_ej1))

# Ecuaciones de D'Alembert (equilibrio en O):
# sum Fx: ROx = m*aGx
ROx = m_ej1 * aGx
# sum Fy: ROy - m*g = m*aGy
ROy = m_ej1 * aGy + m_ej1 * g
# sum M_O: M_motor - m*g*(L/2)*cos(theta) = IO*alpha
M_motor_ej1 = IO_ej1 * alpha_ej1 + m_ej1 * g * (L_ej1/2) * np.cos(theta_ej1)

### Ejercicio 2: Par motor por Potencias Virtuales

Para el mismo mecanismo biela-manivela del analisis anterior, calcular el par motor en la posicion
 $\theta_2 = 90^\circ$ usando exclusivamente el metodo de potencias virtuales.

Verificar con el resultado del analisis inverso completo.

**Desarrollo:**

**Cinematica en $\theta_2 = 90°$:** se resuelven las ecuaciones de cierre del lazo vectorial.

**Potencias virtuales:** el par motor se obtiene del balance de potencias:

$$M_{motor}\,\omega_2 + P_{ext} + P_{grav} + P_{inercia} = 0$$

$$M_{motor} = -\frac{P_{ext} + P_{grav} + P_{inercia}}{\omega_2}$$

**Verificacion:** el resultado coincide con el obtenido por D'Alembert (diferencia del orden de $10^{-14}$ Nm).

In [ ]:
# ── Ejercicio 2: Potencias virtuales en theta2 = 90 deg ──
th2_ej = np.radians(90)

# Resolver cinematica en esa posicion
q0_ej = [0.0, L2 + L3]
res_ej = solve_kinematics(th2_ej, q0_ej)
th3_ej, xB_ej, om3_ej, vB_ej, al3_ej, aB_ej = res_ej


# Aceleraciones de centros de masas
aG2x_ej = -(L2/2) * omega2**2 * np.cos(th2_ej)
aG2y_ej = -(L2/2) * omega2**2 * np.sin(th2_ej)

aAx_ej = -L2 * omega2**2 * np.cos(th2_ej)
aAy_ej = -L2 * omega2**2 * np.sin(th2_ej)
aG3x_ej = aAx_ej - (L3/2) * (al3_ej * np.sin(th3_ej) + om3_ej**2 * np.cos(th3_ej))
aG3y_ej = aAy_ej + (L3/2) * (al3_ej * np.cos(th3_ej) - om3_ej**2 * np.sin(th3_ej))

# Velocidades de centros de masas
vG2x_ej = -(L2/2) * omega2 * np.sin(th2_ej)
vG2y_ej =  (L2/2) * omega2 * np.cos(th2_ej)
vAx_ej = -L2 * omega2 * np.sin(th2_ej)
vAy_ej =  L2 * omega2 * np.cos(th2_ej)
vG3x_ej = vAx_ej - (L3/2) * om3_ej * np.sin(th3_ej)
vG3y_ej = vAy_ej + (L3/2) * om3_ej * np.cos(th3_ej)

# Potencias
P_ext_ej = -F_ext * vB_ej
P_grav_ej = -m2*g*vG2y_ej - m3*g*vG3y_ej
P_inercia_ej = -(m2*(aG2x_ej*vG2x_ej + aG2y_ej*vG2y_ej)
               + m3*(aG3x_ej*vG3x_ej + aG3y_ej*vG3y_ej) + IG3*al3_ej*om3_ej
               + m4*aB_ej*vB_ej)

M_motor_ej2 = -(P_ext_ej + P_grav_ej + P_inercia_ej) / omega2

# Comparar con D'Alembert (interpolacion)
idx_90 = np.argmin(np.abs(theta2_arr - th2_ej))
M_ref = M_motor_dalembert[idx_90]

### Ejercicio 3: Equilibrado estatico de un rotor

Un disco tiene tres masas desequilibradas en el mismo plano:
- $m_1 = 100$ g a $r_1 = 50$ mm, $\theta_1 = 0^\circ$
- $m_2 = 80$ g a $r_2 = 60$ mm, $\theta_2 = 120^\circ$
- $m_3 = 60$ g a $r_3 = 40$ mm, $\theta_3 = 240^\circ$

Determinar la masa de correccion $m_c$ a un radio $r_c = 50$ mm.

**Desarrollo:**

**Desequilibrio total** (suma vectorial en el plano complejo):

$$\sum m_i\,r_i\,e^{j\theta_i} = m_1 r_1 + m_2 r_2\,e^{j\,120°} + m_3 r_3\,e^{j\,240°}$$

**Masa de correccion:** se coloca a $r_c = 50$ mm en el angulo opuesto al desequilibrio:

$$m_c\,r_c\,e^{j\theta_c} = -\sum m_i\,r_i\,e^{j\theta_i}$$

**Verificacion:** el residuo $|\sum m_i r_i e^{j\theta_i} + m_c r_c e^{j\theta_c}| \approx 0$.

In [ ]:
# ── Ejercicio 3: Equilibrado estatico ──
masas_est = [
    (0.100, 0.050, 0),     # m1 [kg], r1 [m], theta1 [deg]
    (0.080, 0.060, 120),
    (0.060, 0.040, 240),
]
rc = 0.050  # m

# Suma vectorial de m*r*e^(j*theta)
sum_mr = sum(m * r * np.exp(1j * np.radians(theta))
             for m, r, theta in masas_est)

# Masa de correccion: mc*rc*e^(j*thetac) = -sum_mr
mc_rc_complex = -sum_mr
mc = np.abs(mc_rc_complex) / rc
thetac = np.degrees(np.angle(mc_rc_complex)) % 360


# Verificacion
check = sum_mr + mc * rc * np.exp(1j * np.radians(thetac))

# ── Diagrama polar ──
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(7, 7))
colors_e = [COLOR_ROJO, COLOR_NARANJA, COLOR_MORADO]
for i, (m, r, theta) in enumerate(masas_est):
    mr_val = m * r
    ax.annotate('', xy=(np.radians(theta), mr_val),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors_e[i], lw=2.5))
    ax.text(np.radians(theta), mr_val * 1.15,
            f'$m_{i+1}r_{i+1}$={mr_val*1e6:.0f}',
            fontsize=10, ha='center', color=colors_e[i])

# Masa de correccion
ax.annotate('', xy=(np.radians(thetac), mc*rc),
            xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=3))
ax.text(np.radians(thetac), mc*rc*1.2,
        f'Corr: {mc*1000:.1f}g\n{thetac:.1f}$^\\circ$',
        fontsize=10, ha='center', color=COLOR_PUNTO, fontweight='bold')

ax.set_title('Equilibrado estatico', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### Ejercicio 4: Aplicacion del Teorema de Fuerzas Vivas

Una barra homogenea ($m=3$ kg, $L=0.8$ m) cae desde el reposo en posicion
 horizontal pivotando en un extremo. Calcular $\omega$ cuando pasa por la vertical.

**Desarrollo:**

**Momento de inercia respecto al pivote:**

$$I_O = \frac{mL^2}{3} = \frac{3 \times 0.8^2}{3} = 0.6400 \text{ kg·m}^2$$

**Teorema de fuerzas vivas** (de horizontal a vertical):

El centro de masas desciende $\Delta z = L/2 = 0.40$ m.

$$W_{grav} = m\,g\,\Delta z = 3 \times 9.81 \times 0.40 = 11.77 \text{ J}$$

$$W_{grav} = \Delta T = \frac{1}{2}\,I_O\,\omega^2$$

$$\omega = \sqrt{\frac{2\,m\,g\,(L/2)}{I_O}} = \sqrt{\frac{2 \times 3 \times 9.81 \times 0.4}{0.64}}$$

$$\boxed{\omega = 6.264 \text{ rad/s}}$$

$$v_{punta} = \omega\,L = 5.011 \text{ m/s}$$

In [ ]:
# ── Ejercicio 4: Teorema de fuerzas vivas ──
m_ej4 = 3.0   # kg
L_ej4 = 0.8   # m
IO_ej4 = m_ej4 * L_ej4**2 / 3  # momento de inercia respecto al pivote

# Estado inicial: horizontal (theta=0), omega=0
# Estado final: vertical (theta=90), omega=?
# El centro de masas baja Delta_z = L/2

# Fuerzas vivas: W_gravedad = Delta_T
# m*g*(L/2) = 1/2 * IO * omega^2
Delta_z = L_ej4 / 2
omega_final = np.sqrt(2 * m_ej4 * g * Delta_z / IO_ej4)


# Animacion: energia vs angulo
theta_range = np.linspace(0, np.pi/2, 100)
# En cada angulo: m*g*(L/2)*sin(theta) = 1/2*IO*omega^2
omega_vs_theta = np.sqrt(2 * m_ej4 * g * (L_ej4/2) * np.sin(theta_range) / IO_ej4)
Ep = m_ej4 * g * (L_ej4/2) * (1 - np.sin(theta_range))  # Ep respecto a vertical
Ek = 0.5 * IO_ej4 * omega_vs_theta**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.degrees(theta_range), omega_vs_theta, color=COLOR_ROJO, lw=2.5)
ax1.set_xlabel(r'$\theta$ (grados)', fontsize=12)
ax1.set_ylabel(r'$\omega$ (rad/s)', fontsize=12)
ax1.set_title('Velocidad angular vs posicion', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(np.degrees(theta_range), Ep, color=COLOR_MORADO, lw=2, label='$E_p$ (potencial)')
ax2.plot(np.degrees(theta_range), Ek, color=COLOR_ROJO, lw=2, label='$E_k$ (cinetica)')
ax2.plot(np.degrees(theta_range), Ep + Ek, color=COLOR_FIJO, lw=2, ls='--', label='$E_{total}$')
ax2.set_xlabel(r'$\theta$ (grados)', fontsize=12)
ax2.set_ylabel('Energia (J)', fontsize=12)
ax2.set_title('Balance energetico', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Catalogo de ejercicios

| # | Tipo | Dificultad | Metodo | Descripcion |
|:--|:-----|:-----------|:-------|:------------|
| 1 | **DCL + D'Alembert** | Media | Vectorial | Dibujar DCL, plantear equilibrio de cada eslabon, resolver sistema lineal |
| 2 | **Reacciones en pares** | Media | Vectorial | Calcular fuerzas en articulaciones a partir del DCL |
| 3 | **Potencias virtuales** | Media | Analitico | Hallar par motor sin necesidad de calcular reacciones |
| 4 | **Con rozamiento** | Alta | Vectorial | Incluir fuerzas de friccion (Amontons-Coulomb) en las ecuaciones |
| 5 | **Equilibrado de rotores** | Media | Complejo | Equilibrado estatico (1 plano) o dinamico (2 planos) |
| 6 | **Analisis dinamico completo** | Alta | Combinado | Cinematica + dinamica inversa + graficas de par motor y reacciones |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Plantilla: Tipo 1 - D'Alembert para mecanismo de 4 barras
# ══════════════════════════════════════════════════════════════════════════

def dalembert_4bar(theta2_val, omega2_val, alpha2_val, params):
    """
    Analisis dinamico inverso de un mecanismo de 4 barras por D'Alembert.
    
    Parametros
    ----------
    theta2_val : float - angulo de entrada (rad)
    omega2_val : float - velocidad angular de entrada (rad/s)
    alpha2_val : float - aceleracion angular de entrada (rad/s^2)
    params : dict con claves:
        L1, L2, L3, L4 : longitudes de las barras
        m2, m3, m4 : masas
        IG2, IG3, IG4 : momentos de inercia
        F_ext, theta_F : fuerza externa (magnitud y angulo)
    
    Retorna
    -------
    dict con M_motor, reacciones, cinematica
    """
    L1 = params['L1']; L2 = params['L2']; L3 = params['L3']; L4 = params['L4']
    m2 = params['m2']; m3 = params['m3']; m4 = params['m4']
    IG2 = params['IG2']; IG3 = params['IG3']; IG4 = params['IG4']
    
    # ── Paso 1: Cinematica de posicion (fsolve) ──
    def pos_eq(q):
        theta3, theta4 = q
        eq1 = L2*np.cos(theta2_val) + L3*np.cos(theta3) - L4*np.cos(theta4) - L1
        eq2 = L2*np.sin(theta2_val) + L3*np.sin(theta3) - L4*np.sin(theta4)
        return [eq1, eq2]
    
    def pos_jac(q):
        theta3, theta4 = q
        return np.array([
            [-L3*np.sin(theta3),  L4*np.sin(theta4)],
            [ L3*np.cos(theta3), -L4*np.cos(theta4)]
        ])
    
    q0 = [theta2_val + 0.5, -(theta2_val + 0.3)]  # estimacion inicial
    q_sol = fsolve(pos_eq, q0, fprime=pos_jac)
    theta3_val, theta4_val = q_sol
    
    # ── Paso 2: Velocidades ──
    J = pos_jac(q_sol)
    b_vel = np.array([
        L2*np.sin(theta2_val)*omega2_val,
        -L2*np.cos(theta2_val)*omega2_val
    ])
    dqdt = np.linalg.solve(J, b_vel)
    omega3, omega4 = dqdt
    
    # ── Paso 3: Aceleraciones ──
    b_acc = np.array([
        L2*np.cos(theta2_val)*omega2_val**2 + L3*np.cos(theta3_val)*omega3**2
        - L4*np.cos(theta4_val)*omega4**2 + L2*np.sin(theta2_val)*alpha2_val,
        L2*np.sin(theta2_val)*omega2_val**2 + L3*np.sin(theta3_val)*omega3**2
        - L4*np.sin(theta4_val)*omega4**2 - L2*np.cos(theta2_val)*alpha2_val
    ])
    d2qdt2 = np.linalg.solve(J, b_acc)
    alpha3, alpha4 = d2qdt2
    
    return {
        'theta3': theta3_val, 'theta4': theta4_val,
        'omega3': omega3, 'omega4': omega4,
        'alpha3': alpha3, 'alpha4': alpha4,
    }

# Ejemplo de uso
params_4bar = {
    'L1': 0.30, 'L2': 0.10, 'L3': 0.25, 'L4': 0.20,
    'm2': 0.5, 'm3': 1.0, 'm4': 0.8,
    'IG2': 0.5*0.10**2/12, 'IG3': 1.0*0.25**2/12, 'IG4': 0.8*0.20**2/12,
    'F_ext': 0, 'theta_F': 0
}

res = dalembert_4bar(np.radians(45), 10.0, 0.0, params_4bar)
for key, val in res.items():
    if 'theta' in key:
    else:

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Plantilla: Tipo 4 - Biela-manivela con rozamiento en la corredera
# ══════════════════════════════════════════════════════════════════════════

mu_d = 0.15  # coeficiente de rozamiento dinamico

M_motor_friction = np.zeros(N_pts)

for i in range(N_pts):
    th2 = theta2_arr[i]
    th3 = theta3_arr[i]
    Ax = L2 * np.cos(th2)
    Ay = L2 * np.sin(th2)
    Bx = xB_arr[i]
    
    # ── Sistema con friccion: 9 eq, 9 incognitas ──
    # La friccion T = mu_d * |N14| * sign(-vB) se acopla con N14
    # Usamos iteracion simple: primero sin friccion, luego con N14 estimado
    
    # Estimacion inicial: resolver sin friccion (ya tenemos N14_arr)
    N14_est = N14_arr[i]
    sign_vB = np.sign(vB_arr[i]) if abs(vB_arr[i]) > 1e-10 else 0.0
    T_friction = -mu_d * abs(N14_est) * sign_vB  # se opone al movimiento
    
    # Re-resolver con la fuerza de friccion incluida en el piston
    A_mat = np.zeros((8, 8))
    b_vec = np.zeros(8)
    
    # Eslabon 2 (igual que antes)
    A_mat[0, 0] = 1.0; A_mat[0, 2] = 1.0
    b_vec[0] = m2 * aG2x[i]
    A_mat[1, 1] = 1.0; A_mat[1, 3] = 1.0
    b_vec[1] = m2 * aG2y[i] + m2 * g
    xg2 = xG2[i]; yg2 = yG2[i]
    A_mat[2, 2] = Ay; A_mat[2, 3] = -Ax; A_mat[2, 7] = 1.0
    b_vec[2] = IG2*alpha2 + m2*(xg2*aG2y[i] - yg2*aG2x[i]) + m2*g*xg2
    
    # Eslabon 3 (igual)
    rG3Ax = xG3[i] - Ax; rG3Ay = yG3[i] - Ay
    rBAx = Bx - Ax; rBAy = -Ay
    A_mat[3, 2] = -1.0; A_mat[3, 4] = 1.0
    b_vec[3] = m3 * aG3x[i]
    A_mat[4, 3] = -1.0; A_mat[4, 5] = 1.0
    b_vec[4] = m3 * aG3y[i] + m3 * g
    A_mat[5, 4] = rBAy; A_mat[5, 5] = -rBAx
    b_vec[5] = IG3*alpha3_arr[i] + m3*(rG3Ax*aG3y[i] - rG3Ay*aG3x[i]) + m3*g*rG3Ax
    
    # Eslabon 4 (CON FRICCION)
    # sum Fx: -R43x - F_ext + T_friction = m4*aG4x
    A_mat[6, 4] = -1.0
    b_vec[6] = m4 * aG4x[i] + F_ext - T_friction  # T_friction al otro lado
    # sum Fy: -R43y + N14 - m4*g = 0
    A_mat[7, 5] = -1.0; A_mat[7, 6] = 1.0
    b_vec[7] = m4 * g
    
    x_sol = np.linalg.solve(A_mat, b_vec)
    M_motor_friction[i] = x_sol[7]

# Comparacion
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(theta2_deg, M_motor_dalembert, color=COLOR_PRINCIPAL, lw=2,
        label='Sin rozamiento')
ax.plot(theta2_deg, M_motor_friction, color=COLOR_ROJO, lw=2, ls='--',
        label=f'Con rozamiento ($\\mu_d = {mu_d}$)')
ax.axhline(y=0, color='gray', ls='--', lw=0.8)
ax.set_xlabel(r'$\theta_2$ (grados)', fontsize=12)
ax.set_ylabel(r'$M_{motor}$ (Nm)', fontsize=12)
ax.set_title('Efecto del rozamiento en el par motor', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

delta_M = M_motor_friction.mean() - M_motor_dalembert.mean()
ax.text(180, M_motor_friction.max()*0.8,
        f'$\\Delta M_{{medio}} = {delta_M:.3f}$ Nm',
        fontsize=12, ha='center',
        bbox=dict(boxstyle='round', facecolor=COLOR_CLARO, alpha=0.5))

plt.tight_layout()
plt.show()

## 14. Resumen y formulas clave

### Fuerzas de inercia (D'Alembert)

| Concepto | Formula |
|:---------|:--------|
| Fuerza de inercia | $\vec{F}_i = -m\,\vec{a}_G$ |
| Momento de inercia | $M_i = -I_G\,\alpha$ |
| Steiner | $I_O = I_G + m\,d^2$ |

### Reacciones en pares

| Par | Incognitas | GDL restringidos |
|:----|:-----------|:-----------------|
| Rotacion | $R_x, R_y$ | 2 |
| Prismatico | $N, M$ (o $N, N'$) | 2 |
| Leva | $N$ | 1 |
| RSD | $N, T$ | 2 |

### Rozamiento (Amontons-Coulomb)

$$F \le \mu_s N \quad (\text{estatico}), \qquad T = \mu_d N \quad (\text{dinamico})$$

### Metodos de resolucion

| Metodo | Ventaja | Desventaja | Ecuaciones |
|:-------|:--------|:-----------|:-----------|
| **D'Alembert** (vectorial) | Todas las reacciones | Sistema grande ($3N$ eq.) | $\sum F = 0$, $\sum M = 0$ por eslabon |
| **Pot. Virtuales** (analitico) | 1 sola ecuacion | Sin reacciones | $\sum P = 0$ |

### Tipos de problema

| Tipo | Dato | Incognita |
|:-----|:-----|:----------|
| **Inverso** | Movimiento $(\theta, \omega, \alpha)$ | Fuerzas / par motor |
| **Directo** | Fuerzas / par motor | Movimiento (integrar ODE) |

### Teorema Fuerzas Vivas

$$W_{\text{total}} = \Delta T = T_f - T_i$$

### Equilibrado de rotores

| Tipo | Condicion |
|:-----|:----------|
| Estatico (1 plano) | $\sum m_k r_k e^{j\theta_k} = 0$ |
| Dinamico (2 planos) | + $\sum m_k r_k z_k e^{j\theta_k} = 0$ |